# Qwen3.8-Flash-Next Coder (IQ1_M) on Kaggle 2x T4 with Strata

Run the cells in order. Stop after cell 3 and check the plan: it must say **Qwen3.8-Flash-Next Coder IQ1_M, 58 GB**.
If it names Swift or IQ2_XS, do not launch (that pack is too big for Kaggle's disk).


## 1. Check the machine

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!free -g | head -2

## 2. Install Strata (first time only)

In [ ]:
!curl -fsSL https://github.com/Niko1221/Strata/releases/latest/download/install.sh -o install.sh && sh install.sh

## 3. PATH, update, dry run

In [ ]:
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!strata --version
!strata update
!strata launch --model iq1_m --family coder --vision off --mtp off --tune skip --plan

## 4. Launch in the background (only once; a second copy would fight over port 8080)

In [ ]:
import subprocess, os

log = open("strata.log", "w")
proc = subprocess.Popen(
    ["strata", "launch", "--model", "iq1_m", "--family", "coder",
     "--context", "32k", "--vision", "off", "--mtp", "off",
     "--tune", "skip", "--access", "local", "--port", "8080"],
    stdout=log, stderr=subprocess.STDOUT,
    env=os.environ.copy(), start_new_session=True,
)
print("started, pid", proc.pid)

## 5. Wait until it is ready
If MB used goes past about 85000, interrupt the cell and run `!pkill -f "strata launch"`.


In [ ]:
import time, subprocess

while proc.poll() is None:
    txt = open("strata.log").read()
    if "Strata is serving" in txt or "Ready: http" in txt:
        print("READY"); break
    used = subprocess.run("du -sm ~/.local/share/strata | cut -f1",
                          shell=True, capture_output=True, text=True).stdout.strip()
    print(f"{txt.strip().splitlines()[-1][:100]}  | {used} MB used (kill before 90000)")
    time.sleep(60)
else:
    print("process exited")
!tail -n 25 strata.log

## 6. Test the model (run twice; the second run gives a more realistic speed)

In [ ]:
import requests, time, json

BASE = "http://127.0.0.1:8080"
model_id = requests.get(f"{BASE}/v1/models", timeout=30).json()["data"][0]["id"]
prompt = "Write a Python function that checks if a string is a palindrome."

payload = {
    "model": model_id,
    "messages": [{"role": "user", "content": prompt}],
    "max_tokens": 400,
    "temperature": 0.6,
    "stream": True,
    "stream_options": {"include_usage": True},
}

print(f"Model: {model_id}\n" + "-" * 40)
t0 = time.time(); first = None; chunks = 0; usage_tokens = None

with requests.post(f"{BASE}/v1/chat/completions", json=payload, stream=True, timeout=1800) as r:
    for line in r.iter_lines(decode_unicode=True):
        if not line or not line.startswith("data:"):
            continue
        chunk = line[5:].strip()
        if chunk == "[DONE]":
            break
        try:
            obj = json.loads(chunk)
        except json.JSONDecodeError:
            continue
        if obj.get("usage"):
            usage_tokens = obj["usage"].get("completion_tokens")
        choices = obj.get("choices") or []
        if not choices:
            continue
        delta = choices[0].get("delta", {})
        text = delta.get("content") or delta.get("reasoning_content") or ""
        if text:
            if first is None:
                first = time.time()
            chunks += 1
            print(text, end="", flush=True)

gen_time = time.time() - (first or t0)
tokens = usage_tokens or chunks
print("\n" + "-" * 40)
print(f"First token: {(first or t0) - t0:.2f}s")
print(f"Speed:       {tokens / max(gen_time, 0.01):.2f} tok/s "
      f"({'exact' if usage_tokens else 'approx'})")

## 7. Draw an SVG with live preview, model name and tok/s
Change `prompt` to anything you like. The file is saved to `/kaggle/working/drawing.svg`.


In [ ]:
import requests, time, json, re
from IPython.display import HTML, display, update_display

BASE = "http://127.0.0.1:8080"
model_id = requests.get(f"{BASE}/v1/models", timeout=30).json()["data"][0]["id"]

prompt = "A cute astronaut floating in space with a dark purple background, small stars, a ringed planet, and a crescent moon. Flat colors, rounded shapes."

payload = {
    "model": model_id,
    "messages": [
        {"role": "system", "content": "Reply with one complete SVG only (viewBox 0 0 400 400), no explanation."},
        {"role": "user", "content": prompt},
    ],
    "max_tokens": 6000,
    "temperature": 0.6,
    "stream": True,
    "stream_options": {"include_usage": True},
}

def extract_svg(text):
    i = text.find("<svg")
    if i == -1:
        return ""
    s = text[i:]
    j = s.find("</svg>")
    return s[: j + 6] if j != -1 else s

def show(state, tokens, tps, elapsed, svg):
    return HTML(f'''
    <div style="font-family:monospace;font-size:13px;margin-bottom:8px">
      <b>Model:</b> {model_id} &nbsp;|&nbsp; <b>State:</b> {state} &nbsp;|&nbsp;
      <b>Tokens:</b> {tokens} &nbsp;|&nbsp; <b>Speed:</b> {tps:.2f} tok/s &nbsp;|&nbsp;
      <b>Time:</b> {elapsed:.0f}s
    </div>
    <div style="width:400px;max-width:100%;background:#fff;border:1px solid #888">{svg}</div>
    ''')

handle = display(show("waiting", 0, 0, 0, ""), display_id=True)

t0 = time.time(); first = None
content = ""; tokens = 0; usage_tokens = None; last_draw = 0

with requests.post(f"{BASE}/v1/chat/completions", json=payload, stream=True, timeout=3600) as r:
    for line in r.iter_lines(decode_unicode=True):
        if not line or not line.startswith("data:"):
            continue
        chunk = line[5:].strip()
        if chunk == "[DONE]":
            break
        try:
            obj = json.loads(chunk)
        except json.JSONDecodeError:
            continue
        if obj.get("usage"):
            usage_tokens = obj["usage"].get("completion_tokens")
        choices = obj.get("choices") or []
        if not choices:
            continue
        delta = choices[0].get("delta", {})
        think = delta.get("reasoning_content") or ""
        piece = delta.get("content") or ""
        if not (think or piece):
            continue
        if first is None:
            first = time.time()
        tokens += 1
        state = "drawing" if piece else "thinking"
        content += piece
        now = time.time()
        if now - last_draw > 0.4:
            tps = tokens / max(now - first, 0.01)
            update_display(show(state, tokens, tps, now - t0, extract_svg(content)), display_id=handle.display_id)
            last_draw = now

end = time.time()
final_tokens = usage_tokens or tokens
tps = final_tokens / max(end - (first or t0), 0.01)
svg = extract_svg(content)
update_display(show("done", final_tokens, tps, end - t0, svg), display_id=handle.display_id)

if "</svg>" in svg:
    open("/kaggle/working/drawing.svg", "w").write(svg)
    print("Saved to /kaggle/working/drawing.svg")
else:
    print("The SVG looks cut off. Raise max_tokens and run again.")

## 8. Stop when done, then end the Kaggle session

In [ ]:
!strata launch --stop